[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/odometry/odometry.ipynb)

# The Most Likely Trajectory from Odometry

This example computes the most likely trajectory of a robot around a lap and the uncertainty of every pose from noisy odometry readings and a loop-closing reading. The robot drives a lap and reads, at every stop, how far it moved since the stop before. Each step reading is slightly off, so chaining them by dead reckoning drifts; when the robot returns to the start and recognises it, one more relative pose reading links the end of the lap back to the start and closes the loop.

Each pose is modelled as a motor, an error in it as a twist perturbation on its right, and the pull on a pose as an intrinsic forque paired with a twist through `twist & forque`. Where the term ***duality map*** here means a linear map between a space and its dual readout space, a reading's uncertainty is a covariance duality map `Twist <- Forque`, and its inverse `Forque <- Twist` is the information map that weighs a mismatch; a motor transports both between frames by its sandwich. Every reading pulls only on the two poses it links, so the Gauss-Newton information operator acts reading by reading without assembling a system matrix, and preconditioned conjugate gradients solve it both for the pose corrections and, in one batched solve over all noise modes, for every pose's marginal covariance.

In the matrix notation of pose-graph nonlinear least squares on $\mathrm{SE}(2)$, each reading $z_k$ between poses $x_i$ and $x_j$ defines a Lie-algebra residual $r_k = \log(z_k^{-1} x_i^{-1} x_j)^\vee$ differentiated through left and right $\mathrm{SE}(2)$ Jacobians and $3 \times 3$ adjoint matrices $\operatorname{Ad}_{x_j^{-1} x_i}$, minimizing $\tfrac{1}{2} \sum_k r_k^{\mathsf T} \Sigma_k^{-1} r_k$ assembles and solves the sparse block normal equations $(J^{\mathsf T} W J)\, \delta = -J^{\mathsf T} W r$, and extracting position confidence ellipses projects $3 \times 3$ diagonal blocks of $(J^{\mathsf T} W J)^{-1}$ through heading-dependent $2 \times 3$ coordinate Jacobians.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import numpy as np

from examples.estimation.odometry import render
from examples.util.animation import save_figure, show_states
from numga import NumpyContext, concatenate
from numga.algebras import PGA2D as ga
from numga.topology import Topology

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Motor = ga.gatype.rotor()                           # Rotor+
Twist = ga.gatype.bivector()                        # Bivector+
# An intrinsic forque pairs with a twist via `twist & forque` to read the objective change the twist makes.
Forque = ga.gatype.antibivector()                   # Antibivector-
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Covariance = ga.gatype((Twist, Forque))             # Twist+ <- Forque-
Information = ga.gatype((Forque, Twist))            # Forque- <- Twist+
Quadric = ga.gatype((Plane, Point))                 # Plane+ <- Point-
n_twist_modes = len(Twist.output_subspace)
# Unit-weight point at the origin carried by each pose motor.
origin = mv.w.dual()                                # [] Point-

## Closed lap and relative pose readings

As the robot drives around a closed planar loop, turning faster and slower twice per lap while slipping sideways and back, each stop records a noisy relative pose reading from the preceding stop, followed by one tight loop-closing reading from the start to the final pose. Each reading is a relative motor corrupted on the right by a twist drawn from its body-frame covariance map `Twist <- Forque`, while the start pose carries a tight prior covariance and the remaining poses carry wide vague priors.

In [ ]:
n_steps = 36
step_length = 0.7
# Planar turn-rate modulation and lateral slip amplitude, each cycling twice per lap.
turn_variation, slip_amplitude = 0.4, 0.1
# Translation and rotation standard deviations for step readings, loop closure, the known start, and vague priors.
reading_std, closing_std = (0.03, 0.01), (0.001, 0.0003)
known_std, vague_std = (0.01, 0.01), (100.0, 30.0)
sigmas = 2.0

In [ ]:
def body_covariance(translation_std: float, rotation_std: float) -> Covariance:
    """Translation variance `translation_std**2` and rotation variance `rotation_std**2` about the body origin form an isotropic body-frame covariance (`[] Twist+ <- Forque-`)."""
    basis_twists = mv(Twist, np.eye(n_twist_modes))                             # [n_twist_modes] Twist+
    # A basis twist generates a rotation when its reverse norm squared is one and a translation when it is zero.
    rotational = basis_twists.scalar_norm_squared()                             # [n_twist_modes] Scalar+
    variances = translation_std**2 + (rotation_std**2 - translation_std**2) * rotational  # [n_twist_modes] Scalar+
    return (basis_twists * (basis_twists & Forque) * variances).sum(axis=0)     # [] Twist+ <- Forque-


def covariance_modes(uncertainty: Covariance) -> Twist:
    """Generalized eigenvectors of the readout form `uncertainty & Forque` map through `uncertainty` (`[...] Twist+ <- Forque-`) to twist modes (`[..., n_twist_modes] Twist+`) whose outer products sum back to `uncertainty`."""
    spread = uncertainty & Forque                                               # [...] Scalar+ <- (Forque-, Forque-)
    _, forques = spread.eigh(spread)                                            # [..., n_twist_modes] Forque-
    return uncertainty[..., None](forques)                                      # [..., n_twist_modes] Twist+


rng = np.random.default_rng(9)
phase = 2 * np.pi * np.arange(n_steps) / n_steps
turn = mv.xy * (1 + turn_variation * np.sin(2 * phase)) * 2 * np.pi / n_steps   # [n_steps] Twist+
sideways = mv.yw * slip_amplitude * np.cos(2 * phase)                           # [n_steps] Twist+
steps = ((turn + sideways + mv.xw * step_length) * 0.5).exp()                   # [n_steps] Rotor+
# Body-frame steps compose on the right; reversing the cumulative left product of reverses gives the cumulative right product.
edges = np.stack([np.append(np.arange(n_steps), 0), np.append(np.arange(1, n_steps + 1), n_steps)], axis=-1)
topology = Topology(n_steps + 1, edges)
V, E = topology.n_elements
tails, heads = topology.elements[1].T                                           # [E], [E]
truth = concatenate([mv.rotor()[None], steps.reverse().cumprod(axis=0).reverse()])  # [V] Rotor+

noises = concatenate([body_covariance(*reading_std) * np.ones(n_steps), body_covariance(*closing_std)[None]])  # [E] Twist+ <- Forque-
noise_modes = covariance_modes(noises)                                          # [E, n_twist_modes] Twist+
errors = (noise_modes * rng.normal(size=noise_modes.shape)).sum(axis=-1)        # [E] Twist+
error_motors = (errors * 0.5).exp()                                             # [E] Rotor+
readings = (truth[tails].inverse() * truth[heads]) * error_motors               # [E] Rotor+
priors = concatenate([body_covariance(*known_std)[None], body_covariance(*vague_std) * np.ones(n_steps)])  # [V] Twist+ <- Forque-
weights = noises.inverse()                                                      # [E] Forque- <- Twist+
anchor_weights = priors.inverse()                                               # [V] Forque- <- Twist+

## Dead reckoning and position confidence quadrics

Chaining the step readings from the known start drifts steadily across the lap, and pose uncertainty grows along with it: each step's noise, pushed to the world frame by the pose where it enters, accumulates with every earlier step's noise and pulls back to each pose's body frame as its covariance. Carrying body twists to world space and taking `Twist.commutator(positions)` maps twists to point displacements, whose `.adjugate()` pulls displacement-measuring planes back to twist readout forques and yields each pose's implicit position confidence ellipse.

In [ ]:
def dead_reckoning_covariances(poses: Motor, noises: Covariance, priors: Covariance) -> Covariance:
    """Propagating `priors[:1]` forward through each step's noise in `noises` gives the dead-reckoned body-frame covariances (`[V] Twist+ <- Forque-`)."""
    step_noises = concatenate([priors[:1], noises[:V - 1]])                     # [V] Twist+ <- Forque-
    world_uncertainty = (poses >> step_noises(poses << Forque)).cumsum(axis=0)  # [V] Twist+ <- Forque-
    return poses << world_uncertainty(poses >> Forque)                          # [V] Twist+ <- Forque-


def position_ellipsoids(poses: Motor, uncertainty: Covariance, sigmas: float) -> Quadric:
    """Pushing `uncertainty` (`[...] Twist+ <- Forque-`) through the body-twist motion of `poses >> origin` gives the implicit confidence quadrics (`[...] Plane+ <- Point-`) at `sigmas` standard deviations."""
    positions = poses >> origin                                                 # [...] Point-
    # Body-frame twists carried to world space displace each pose's position by their commutator with it.
    shift = Twist.commutator(positions)(poses >> Twist)                         # [...] Point- <- Twist+
    # Pulling displacement-measuring planes back through that motion gives the corresponding twist readout forques.
    readout = shift.adjugate()                                                  # [...] Forque- <- Plane+
    # Adding the pushed-forward position covariance to the position's own dyad forms its second-moment dual quadric.
    moment = positions * (Plane & positions) + shift(uncertainty(readout))      # [...] Point- <- Plane+
    return moment.inverse() - (1 + sigmas**2) * mv.w * (mv.w & Point)           # [...] Plane+ <- Point-


dead_reckoning = concatenate([truth[:1], truth[0] * readings[:-1].reverse().cumprod(axis=0).reverse()])  # [V] Rotor+
anchors = concatenate([truth[:1], dead_reckoning[1:]])                          # [V] Rotor+
reckoned_uncertainty = dead_reckoning_covariances(dead_reckoning, noises, priors)  # [V] Twist+ <- Forque-
reckoned_ellipsoids = position_ellipsoids(dead_reckoning, reckoned_uncertainty, sigmas)  # [V] Plane+ <- Point-
save_figure(render.dead_reckoning_figure(truth, dead_reckoning, reckoned_ellipsoids, dead_reckoning[[tails[-1], heads[-1]]]), "odometry_dead_reckoning");

The grey curve traces the true closed lap from the dark start marker, the dashed orange curve and orange ellipses show the drifting dead-reckoned trajectory and its two-standard-deviation position uncertainty, and the green segment marks the gap exposed by the loop-closing reading between the start and the final dead-reckoned pose.

## Mismatches and the matrix-free Gauss-Newton operator

A reading's mismatch is the logarithmic twist `(readings.inverse() * relative_motors).log() * 2` from the measured step to the step implied by its endpoint poses. Weighing that twist by the reading's information map produces a forque pulling on the head pose, while the opposite forque carried across the relative motor pulls on the tail pose; summing reading and anchor pulls gives the objective gradient. Applying the same pull to linearized reading twist perturbations gives the Gauss-Newton information operator without assembling a system matrix, and summing what each pose's perturbation causes on its own gives the block-diagonal preconditioner.

In [ ]:
def pose_forques(relative_motors: Motor, reading_twists: Twist, anchor_twists: Twist) -> Forque:
    """Weighing `reading_twists` (`[..., E] Twist+`) and `anchor_twists` (`[..., V] Twist+`) across the pose graph sums the net forques (`[..., V] Forque-`) on every pose."""
    head_forques = weights(reading_twists)                                      # [..., E] Forque-
    anchor_forques = anchor_weights(anchor_twists)                              # [..., V] Forque-
    # Each reading pulls on its head directly and on its tail with the opposite forque carried across the relative motor.
    tail_forques = -(relative_motors >> head_forques)                           # [..., E] Forque-
    total_forques = anchor_forques.at[..., heads].add(head_forques)             # [..., V] Forque-
    return total_forques.at[..., tails].add(tail_forques)                       # [..., V] Forque-


def objective_gradient(poses: Motor) -> Forque:
    """Logarithmic mismatches between `poses` (`[V] Motor+`), `readings` (`[E] Motor+`), and `anchors` (`[V] Motor+`) exert the pose-graph objective gradient (`[V] Forque-`)."""
    relative_motors = poses[tails].inverse() * poses[heads]                     # [E] Rotor+
    reading_mismatches = (readings.inverse() * relative_motors).log() * 2       # [E] Twist+
    anchor_mismatches = (anchors.inverse() * poses).log() * 2                   # [V] Twist+
    return pose_forques(relative_motors, reading_mismatches, anchor_mismatches)  # [V] Forque-

In [ ]:
def information_product(poses: Motor, twists: Twist) -> Forque:
    """Applying the Gauss-Newton information operator at `poses` (`[V] Motor+`) maps pose perturbations `twists` (`[..., V] Twist+`) to the resulting forques (`[..., V] Forque-`)."""
    # Each reading sees the twist perturbation at its head minus the perturbation at its tail carried to the head.
    relative_motors = poses[tails].inverse() * poses[heads]                     # [E] Rotor+
    reading_twists = twists[..., heads] - (relative_motors << twists[..., tails])  # [..., E] Twist+
    return pose_forques(relative_motors, reading_twists, twists)                # [..., V] Forque-


def diagonal_information(poses: Motor) -> Information:
    """Summing each pose's own incident reading and anchor weights gives the per-pose block-diagonal preconditioner (`[V] Forque- <- Twist+`)."""
    relative_motors = poses[tails].inverse() * poses[heads]                     # [E] Rotor+
    # Perturbing a reading's head alone meets its weight directly; perturbing its tail alone meets the weight carried back across the relative motor.
    tail_weights = relative_motors >> weights(relative_motors << Twist)         # [E] Forque- <- Twist+
    pose_weights = anchor_weights.at[heads].add(weights)                        # [V] Forque- <- Twist+
    return pose_weights.at[tails].add(tail_weights)                             # [V] Forque- <- Twist+

## Preconditioned conjugate gradients and Gauss-Newton steps

Preconditioned conjugate gradients solve the linear normal equations for twist corrections across arbitrary leading batch axes, reaching the exact linear solution in at most the number of scalar unknowns. Because the pose-graph objective is nonlinear on the motor manifold, each Gauss-Newton iteration solves the linearized system at the current poses and multiplies the resulting twist motor onto the right of every pose.

In [ ]:
def conjugate_gradient_solves(poses: Motor, forques: Forque) -> Iterator[Twist]:
    """Yields preconditioned conjugate-gradient twist corrections (`[..., V] Twist+`) converging on the Gauss-Newton normal-equation solution."""
    preconditioner = diagonal_information(poses)                                # [V] Forque- <- Twist+
    correction = 0 * preconditioner.solve(forques)                              # [..., V] Twist+
    residual = forques                                                          # [..., V] Forque-
    direction = preconditioner.solve(residual)                                  # [..., V] Twist+
    residual_energy = (direction & residual).sum(axis=-1)                       # [...] Scalar+
    # Conjugate gradients reach the exact linear solution in at most the number of scalar unknowns.
    for _ in range(n_twist_modes * V):
        direction_forques = information_product(poses, direction)               # [..., V] Forque-
        step = (residual_energy / (direction & direction_forques).sum(axis=-1))[..., None]
        correction = correction + direction * step                              # [..., V] Twist+
        residual = residual - direction_forques * step                          # [..., V] Forque-
        preconditioned_residual = preconditioner.solve(residual)                # [..., V] Twist+
        next_energy = (preconditioned_residual & residual).sum(axis=-1)         # [...] Scalar+
        direction = preconditioned_residual + direction * (next_energy / residual_energy)[..., None]  # [..., V] Twist+
        residual_energy = next_energy
        yield correction


def gauss_newton_iterates(poses: Motor, damping: float, n_iterations: int) -> Iterator[Motor]:
    """Yields the updated pose trajectory (`[V] Motor+`) across `n_iterations` damped Gauss-Newton steps."""
    for _ in range(n_iterations):
        *_, correction = conjugate_gradient_solves(poses, -objective_gradient(poses))  # [V] Twist+
        motor_step = (correction * (0.5 * damping)).exp()                       # [V] Rotor+
        poses = poses * motor_step                                              # [V] Rotor+
        yield poses

## Most likely trajectory and marginal covariances

Full Gauss-Newton steps from dead reckoning converge rapidly onto the most likely closed trajectory. Each pose's marginal covariance is its diagonal block in the inverse of the Gauss-Newton information operator: solving in one batch for the pose errors induced by every unit mode of reading and anchor noise and summing their outer products `pose_errors * (pose_errors & Forque)` recovers all pose covariances together.

In [ ]:
n_full_steps = 3

In [ ]:
def marginal_covariances(poses: Motor) -> Covariance:
    """Solving in one batch for the pose errors induced by every reading and anchor noise mode gives the marginal covariances (`[V] Twist+ <- Forque-`)."""
    relative_motors = poses[tails].inverse() * poses[heads]                     # [E] Rotor+
    noise_modes = concatenate([covariance_modes(noises), covariance_modes(priors)])  # [E + V, n_twist_modes] Twist+
    unit_modes = noise_modes[..., None] * np.eye(E + V)[:, None]                # [E + V, n_twist_modes, E + V] Twist+
    mode_forques = pose_forques(relative_motors, unit_modes[..., :E], unit_modes[..., E:])  # [E + V, n_twist_modes, V] Forque-
    *_, pose_errors = conjugate_gradient_solves(poses, mode_forques)            # [E + V, n_twist_modes, V] Twist+
    return (pose_errors * (pose_errors & Forque)).sum(axis=(0, 1))              # [V] Twist+ <- Forque-


*_, poses = gauss_newton_iterates(dead_reckoning, 1.0, n_full_steps)            # [V] Rotor+
uncertainty = marginal_covariances(poses)                                       # [V] Twist+ <- Forque-
save_figure(render.trajectory_figure(truth, dead_reckoning, reckoned_ellipsoids, poses, position_ellipsoids(poses, uncertainty, sigmas)), "odometry");

The blue trajectory and blue ellipses show the recovered poses and their posterior two-standard-deviation position uncertainty over the grey true lap and dashed orange dead reckoning; uncertainty peaks halfway around the lap, farthest from the anchored start and loop closure, and pinches back down where the loop closes.

## Damped loop closure

Taking a fraction `damping` of each Gauss-Newton twist correction and relinearizing at the updated poses pulls the open end of the trajectory smoothly back to the start. Because each damped step removes `damping` of the remaining linearized error, the excess covariance above the posterior marginal covariance shrinks by `(1 - damping)**2` per step.

In [ ]:
damping = 0.3
n_damped_steps = 10
frame_duration_ms = 333

In [ ]:
def relaxation_states(iterates: Iterator[Motor], uncertainty: Covariance, damping: float) -> Iterator[tuple[Motor, Covariance]]:
    """Yields `(poses, uncertainty)` (`[V] Rotor+`, `[V] Twist+ <- Forque-`) as the pose covariances relax toward the posterior marginals across damped steps."""
    for poses in iterates:
        posterior = marginal_covariances(poses)                                 # [V] Twist+ <- Forque-
        uncertainty = posterior + (uncertainty - posterior) * (1 - damping)**2  # [V] Twist+ <- Forque-
        yield poses, uncertainty


closing_states = relaxation_states(gauss_newton_iterates(dead_reckoning, damping, n_damped_steps), reckoned_uncertainty, damping)
recorded = show_states(
    closing_states,
    render.closing_animation,
    truth,
    dead_reckoning,
    reckoned_ellipsoids,
    partial(position_ellipsoids, sigmas=sigmas),
    name="odometry",
    duration_ms=frame_duration_ms,
    n_frames=n_damped_steps,
    stepped=True,
);

Across the ten damped Gauss-Newton steps, the blue trajectory closes the drift gap back onto the dark start marker while the blue position ellipses contract from the wide orange dead-reckoned spread onto the tight loop-closed posterior uncertainty.

In [ ]:
# checks
# Full Gauss-Newton drives the covariance-weighted squared gradient down more than twelve orders of magnitude,
# every posterior covariance stays symmetric under `.adjugate()`, and damped closure shrinks end-of-lap uncertainty.
dead_gradient = objective_gradient(dead_reckoning)                              # [V] Forque-
optimum_gradient = objective_gradient(poses)                                    # [V] Forque-
assert bool((uncertainty(optimum_gradient) & optimum_gradient).max() < 1e-12 * (uncertainty(dead_gradient) & dead_gradient).max())

unit_info = body_covariance(1.0, 1.0).inverse()                                 # [] Forque- <- Twist+
symmetry_map = (uncertainty.adjugate() - uncertainty)(unit_info)                # [V] Twist+ <- Twist+
assert bool((symmetry_map(symmetry_map).trace(0, 1).abs() < 1e-20).all())

_, final_damped_uncertainty = recorded[-1]
assert bool(final_damped_uncertainty[-1](unit_info).trace(0, 1) < 0.01 * reckoned_uncertainty[-1](unit_info).trace(0, 1))